# Make metadata

Read `Campioni meta_REFLUI.xlsx` and try to create a metadata file, and then split 
into bacteria and fungi metadata files. Start by import some stuff

In [ ]:
import re

import pandas as pd

from utils import find_project_root

`find_project_root` is an utility function to detect the absolute path of the current 
directory: I use it (like the *R* `here` package) to avoid to use absolute paths in
the code. Determine where the data files are:

In [ ]:
data_dir = find_project_root() / 'data'

Time to read the original data file:

In [ ]:
# Read the dataframe from the file
df = pd.read_excel(data_dir / 'Campioni meta_REFLUI.xlsx')

# Display the dataframe
df.head()

In [ ]:
df.info()

There a are some empty columns and rows, let's remove them. Cast integer values 
to integers (but use `Int64` to allow for missing values):

In [ ]:
new_df = df.drop(df.columns[0], axis=1)  # Remove the first column
new_df.drop(range(39, len(new_df)), axis=0, inplace=True)  # Remove rows from 39th row until the end

new_df['Unnamed: 1'] = new_df['Unnamed: 1'].astype('Int64')
new_df['Unnamed: 2'] = new_df['Unnamed: 2'].astype('Int64')
new_df.head()

Ensure that column names and values have no leading or trailing spaces:

In [ ]:
df_stripped = new_df.apply(lambda x: x.str.strip() if x.dtype == "object" else x)
df_stripped.columns = df_stripped.columns.str.strip()
df_stripped.head()

Rename columns:

In [ ]:
new_df = df_stripped.rename(columns={
    "Unnamed: 1": "sample_group",
    "Unnamed: 2": "sample_id",
    "Sample": "sample_name",
    "Unnamed: 4": "description",
    "CONCENTRATION (ng/ul)": "concentration",
    "A260\nA280": "A260-A280",
    "A260\nA230": "A260-A230"})
new_df.head()


There are cells with missing values: in the original `.xlsx` there are *united*, now
only the first of the group has values. Let's fill the missing values with the latests
seen value:

In [ ]:
filled_df = new_df.ffill()
filled_df.head()

Here some *spaghetti* code to define some new columns relying on `description`:

In [ ]:
# Define patterns for date and condition extraction
date_pattern = r'TQ\s(\w+\s\d{2})'

condition_patterns = {
    'cresciuto in multicoltivatore con Chlorella sorokiniana': r'cresciuto in multicoltivatore con Chlorella sorokiniana',
    'cresciuto in multicoltivatore': r'cresciuto in multicoltivatore(?! con Chlorella sorokiniana)',
    'fatta crescere la duckweed L.m. 5500 per 7 gg': r'fatta crescere la duckweed L\.m\. 5500 per 7 gg',
    'mantenuto 7 gg nelle stesse condizioni di crescita della prova con la 5500': r'mantenuto 7 gg nelle stesse condizioni di crescita della prova con la 5500'
}
default_condition = 'default'

replica_pattern = r'Replica\s(\d)'

# Extract the date
filled_df['date'] = filled_df['description'].str.extract(date_pattern)

# Extract the condition
def extract_condition(description):
    for condition, pattern in condition_patterns.items():
        if re.search(pattern, description, re.IGNORECASE):
            return condition
    return default_condition

filled_df['condition'] = filled_df['description'].apply(extract_condition)

# Extract the replica number
filled_df['replica'] = filled_df['description'].str.extract(replica_pattern)
filled_df.head()

Nextflow requires that sample names should start with a letter. Rename `sample_name` to `sampleID` (like nextflow is used to). Then move `sampleID` to
the first column:

In [ ]:
# Rename the column
filled_df.rename(
    columns={
        'sample_name': 'sampleID',
        'sample_id': 'sample_number'
    },
    inplace=True
)

# Reorder the columns to make 'sampleID' the first column
cols = ['sampleID'] + [col for col in filled_df.columns if col != 'sampleID']
filled_df = filled_df[cols]

filled_df.head()

Here we duplicate the table for *bacteria* and *fungi*: the *bacteria* `sampleID`
where renamed since there were added after the *fungi* `sampleID` (incremented by 39):

In [ ]:
# Duplicate the filled table
df_fungi = filled_df.copy()
df_bacteria = filled_df.copy()

# Update the target column for the copied rows
df_bacteria['target'] = 'bacteria'

# Update the target column for the original table
df_fungi['target'] = 'fungi'

# update sample ids and combine into a final dataframe
df_bacteria['sample_number'] = df_bacteria['sample_number'].apply(lambda x: x + 39)

final_df = pd.concat([df_fungi, df_bacteria])
final_df

Time to write the metadata files:

In [ ]:
final_df.to_excel(data_dir / 'metadata.xlsx', index=False)
df_bacteria.to_csv(data_dir / 'metadata_bacteria.csv', index=False)
df_fungi.to_csv(data_dir / 'metadata_fungi.csv', index=False)